# Effect of Grid Electricity on Small-Firm Revenue: Matching and Instrumental Variables
**Question:** Does connecting to the electricity grid raise revenue for small and medium enterprises (SMEs)?

**Data:** Course-provided instructional dataset of 2,500 SMEs (not a real program), from a Program Evaluation course assignment. Variables: `grid_connection` (treatment), `firm_revenue` (outcome), `grid_subsidy_amount` (randomly assigned subsidy, used as the instrument), and owner/firm characteristics (`owner_older40`, `owner_male`, `num_livestock`, `years_edu`, `num_employees`).

**Approach:** (1) show that connected and unconnected firms differ, (2) compare naive and exact-matching estimates, (3) use the randomized subsidy as an instrument in a two-stage least squares (IV) model.

**Main results:** The naive difference is about $12,992; exact matching gives about $11,645 (will vary slightly with the random seed); the IV estimate is about $7,900 (95% CI $7,074 to $8,726). The naive estimate overstates the IV estimate by about 64%, consistent with positive selection into connection. The instrument is strong (first-stage F = 170.5).

In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
from scipy import stats

SEED = 42  # fixed so the matching step is reproducible

In [3]:
df = pd.read_csv("ps2_data_Spring26.csv")
df.head()

,owner_older40,owner_male,num_livestock,years_edu,num_employees,grid_subsidy_amount,grid_connection,firm_revenue
0,1,0,2,9,1,150,0,11055.950
1,0,1,2,10,3,150,1,19540.050
2,0,1,0,6,2,200,0,10091.050
3,0,0,5,8,1,200,0,6820.209
4,0,1,0,9,2,50,0,10722.090


## 1. Covariate balance: who gets connected to the grid?
Compare firms with and without a grid connection on observable characteristics. Large, statistically significant differences mean connection is not randomly assigned, so a simple comparison of revenue would mix the effect of electricity with pre-existing differences between firms.

In [ ]:
def balance_table(df, covariates, treatment_col):
    rows = []
    for var in covariates:
        treated = df[df[treatment_col] == 1][var].dropna()
        control = df[df[treatment_col] == 0][var].dropna()

        t_stat, p_val = stats.ttest_ind(treated, control)

        rows.append(
            {
                "Variable": var,
                "Mean (Treated)": treated.mean().round(3),
                "Mean (Control)": control.mean().round(3),
                "Difference": (treated.mean() - control.mean()).round(3),
                "P-value": p_val.round(3),
            }
        )

    return pd.DataFrame(rows)


# Usage
covariates = [
    "owner_older40",
    "owner_male",
    "num_livestock",
    "years_edu",
    "num_employees",
    "grid_subsidy_amount",
]
table = balance_table(df, covariates, treatment_col="grid_connection")
print(table)

              Variable  Mean (Treated)  Mean (Control)  Difference  P-value
0        owner_older40           0.538           0.152       0.386    0.000
1           owner_male           0.646           0.451       0.196    0.000
2        num_livestock           2.405           2.337       0.068    0.419
3            years_edu           7.408           6.820       0.588    0.000
4        num_employees           2.098           1.977       0.121    0.095
5  grid_subsidy_amount         181.040         139.760      41.280    0.000


## 2. Naive and exact-matching estimates
The naive estimate is the raw difference in mean revenue between connected and unconnected firms. The matching estimate pairs each connected firm with an unconnected firm that is identical on the matched variables, then compares revenue (an estimate of the average effect on connected firms, the ATT). Matching can only adjust for the variables I observe and match on; unobserved differences (e.g., owner ambition, access to credit) can still bias the result.

In [ ]:
# Exact match (with replacement) on owner age group, gender, livestock count, and years of education
match_cols = ["owner_older40", "owner_male", "num_livestock", "years_edu"]

treated = df[df["grid_connection"] == 1]
control = df[df["grid_connection"] == 0]

matched_controls = []
n_unmatched = 0
for _, t_row in treated.iterrows():
    # Find control units that match exactly on all match_cols
    mask = pd.Series([True] * len(control), index=control.index)
    for col in match_cols:
        mask &= control[col] == t_row[col]

    candidates = control[mask]
    if len(candidates) > 0:
        matched_controls.append(
            candidates.sample(1, random_state=SEED)
        )  # random pick if multiple matches
    else:
        n_unmatched += 1

matched_control_df = pd.concat(matched_controls)
matched_df = pd.concat([treated, matched_control_df])

att = (
    matched_df[matched_df["grid_connection"] == 1]["firm_revenue"].mean()
    - matched_df[matched_df["grid_connection"] == 0]["firm_revenue"].mean()
)
print(f"Treated firms with no exact match: {n_unmatched} of {len(treated)}")
print(f"ATT estimate: {att:,.0f}")

# naive estimate
naive_effect = (
    df[df["grid_connection"] == 1]["firm_revenue"].mean()
    - df[df["grid_connection"] == 0]["firm_revenue"].mean()
)
print(f"naive estimate: {naive_effect:,.0f}")

## 3. Instrumental variables, step 1: first stage
Use the randomly assigned grid-connection subsidy as an instrument for connection. The first stage checks **relevance**: does a larger subsidy actually make a firm more likely to connect? A first-stage F-statistic well above 10 is the usual rule of thumb for a strong instrument.

In [ ]:
first_stage = smf.ols("grid_connection ~ grid_subsidy_amount", data=df).fit()
print(first_stage.summary())

                            OLS Regression Results                            
Dep. Variable:        grid_connection   R-squared:                       0.064
Model:                            OLS   Adj. R-squared:                  0.064
Method:                 Least Squares   F-statistic:                     170.5
Date:                Mon, 27 Apr 2026   Prob (F-statistic):           9.52e-38
Time:                        22:39:59   Log-Likelihood:                -1372.3
No. Observations:                2500   AIC:                             2749.
Df Residuals:                    2498   BIC:                             2760.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept               0.0177    

## 4. Instrumental variables, step 2: reduced form
The reduced form regresses revenue directly on the subsidy (the intention-to-treat effect of offering a subsidy). Dividing it by the first-stage coefficient gives the Wald estimate of the effect of connection for firms whose behavior the subsidy changes.

In [14]:
reduced_form = smf.ols("firm_revenue ~ grid_subsidy_amount", data=df).fit()
print(reduced_form.summary())

                            OLS Regression Results                            
Dep. Variable:           firm_revenue   R-squared:                       0.018
Model:                            OLS   Adj. R-squared:                  0.018
Method:                 Least Squares   F-statistic:                     45.54
Date:                Mon, 27 Apr 2026   Prob (F-statistic):           1.85e-11
Time:                        23:07:29   Log-Likelihood:                -25306.
No. Observations:                2500   AIC:                         5.062e+04
Df Residuals:                    2498   BIC:                         5.063e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept            9388.4633    

## 5. Two-stage least squares
Estimate the effect of connection on revenue using the subsidy as the instrument, controlling for owner age group, gender, education, and number of employees. Standard errors are heteroskedasticity-robust.

In [ ]:
# Add constant to exogenous variables
X_exog = sm.add_constant(
    df[["owner_older40", "owner_male", "years_edu", "num_employees"]]
)

model = IV2SLS(
    dependent=df["firm_revenue"],
    exog=X_exog,
    endog=df["grid_connection"],
    instruments=df["grid_subsidy_amount"],
).fit()

print(model.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:           firm_revenue   R-squared:                      0.8390
Estimator:                    IV-2SLS   Adj. R-squared:                 0.8387
No. Observations:                2500   F-statistic:                    5453.2
Date:                Tue, Apr 28 2026   P-value (F-stat)                0.0000
Time:                        15:03:17   Distribution:                  chi2(5)
Cov. Estimator:                robust                                         
                                                                              
                                Parameter Estimates                                
                 Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-----------------------------------------------------------------------------------
const               4233.0     221.40     19.119     0.0000      3799.1      4667.0
owner_older40       3769.8     2

## Interpretation and limitations
- **What the IV estimate means:** it is a local average treatment effect (LATE): the effect for firms whose connection decision responds to the subsidy, not necessarily for all firms.
- **Exclusion restriction:** the IV estimate requires that the subsidy affects revenue only through grid connection. This could fail if the subsidy works as a cash transfer (a liquidity effect) that helps firms independent of electricity.
- **Randomization:** the subsidy is assumed to be randomly assigned; a balance check of subsidy amount against owner and firm characteristics would support this and is not included here.
- **Matching:** only observed covariates are matched on, matching is with replacement, and treated firms without an exact match are counted in the diagnostic above. Results depend on the random draw among eligible controls.
- **External validity:** this is a simulated course dataset, so the numbers illustrate the methods and are not evidence about any real program.